# Dataset Exploration

This notebook is a practical sandbox for understanding the Hugging Face `datasets` library.

You will learn how to:

1. Load a dataset from an online source and from a local file
2. Review dataset features and schema
3. Use `shuffle()` and `map()` to transform data
4. Select smaller training and evaluation subsets for experiments


<a href="https://colab.research.google.com/github/ned1313/Fine-tuning-and-Optimizing-Small-Language-Models/blob/main/notebooks/datasets.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Google Colab prep

If you are running this notebook in Google Colab, run the code block below to install the necessary packages.

In [ ]:
%pip install datasets

## Load a dataset from Hugging Face Hub

In this example we load the MRPC dataset from Hugging Face.

In [ ]:
from datasets import load_dataset

online_ds = load_dataset("nyu-mll/glue","mrpc")

In [ ]:
online_ds

## Review dataset features and sample rows

Inspect schema, column names, and sample records from both datasets.

Using the code cell below try some of the following expressions to explore the datasets:

```python
online_ds
online_ds["train"]
online_ds["train"].features
online_ds["train"].column_names
online_ds["train"][0]
online_ds["train"]["sentence1"][:5]
```

In [ ]:
online_ds["train"]["sentence1"][:5]

## Use `shuffle()` and `map()`

`shuffle()` randomizes example order and `map()` lets you transform or add fields.

In [ ]:
def enrich_example(example: dict) -> dict:
    text = example["sentence1"] + " " + example["sentence2"]
    return {
        "combined_text": text,
        "char_count": len(text),
        "word_count": len(text.split()),
    }

online_train_enriched = online_ds["train"].map(enrich_example)

online_train_enriched.to_pandas().head(5)

In [ ]:
online_train_combined = online_ds["train"].map(enrich_example, remove_columns=online_ds["train"].column_names)

online_train_combined.to_pandas().head(5)

In [ ]:
shuffled_ds = online_train_combined.shuffle(seed=42)

shuffled_ds.to_pandas().head(5)

## Select portions for training and evaluation

For fast experiments, take a smaller slice from a split.

You can do this by shuffling and selecting fixed ranges or by using `train_test_split()`.

In [ ]:
seed = 42
train_take = 2000
eval_take = 500

online_train_subset = online_ds["train"].shuffle(seed=seed).select(range(train_take))
online_eval_subset = online_ds["test"].shuffle(seed=seed).select(range(eval_take))

print("Manual subset sizes")
print("train:", len(online_train_subset))
print("eval:", len(online_eval_subset))


In [ ]:

split_result = online_ds["train"].shuffle(seed=seed).train_test_split(test_size=0.2, seed=seed)
print("\ntrain_test_split sizes")
print("train:", len(split_result["train"]))
print("eval:", len(split_result["test"]))


## Load a dataset from a local file

This cell creates a small local JSONL file so the notebook is self-contained.

If you already have a local CSV or JSONL file, replace `local_data_path` with your own path.

In [ ]:
from pathlib import Path
import json

REPO_ROOT = Path.cwd().resolve().parent
local_data_path = REPO_ROOT / "datasets" / "sample_sentiment.jsonl"

if not local_data_path.exists():
    local_rows = [
        {"text": "I loved the lesson and examples.", "label": 1, "source": "human"},
        {"text": "This section was confusing.", "label": 0, "source": "human"},
        {"text": "Great pacing and clear explanations.", "label": 1, "source": "human"},
        {"text": "The audio quality dropped midway.", "label": 0, "source": "human"},
        {"text": "Very practical and easy to follow.", "label": 1, "source": "human"},
    ]
    with local_data_path.open("w", encoding="utf-8") as file_obj:
        for row in local_rows:
            file_obj.write(json.dumps(row) + "\n")

file_dataset = load_dataset("json", data_files={"train": str(local_data_path)})

file_dataset